# Credit Card Fraud – End‑to‑End Playbook (EDA → Data Quality → Modeling → Evaluation → Persistenz)

**Was du hier bekommst:** Der frühere Monolith (`credit_card_fraud_eda_and_model.py`) ist in **klare, ausführbare Schritte** zerlegt.
Jeder Abschnitt erklärt kurz **warum** wir das tun und führt den **Code** separat aus.


## Inhaltsverzeichnis
1. Setup & Installation (optional)
2. Imports & Konfiguration
3. Daten laden
4. Schnellüberblick (Head, Info, Describe)
5. Datenqualität (Missing, Duplikate, Dtypes)
6. Zielverteilung & Klassenungleichgewicht
7. Feature Engineering (Hour)
8. Visuelle EDA (Plots werden gespeichert)
9. Kompakter Datenqualitäts‑Report
10. Modell‑Vorbereitung (X/y)
11. Train/Test Split
12. Skalierung (StandardScaler)
13. Klassenbalancierung (SMOTE)
14. Baseline‑Modelle trainieren (LogReg, RandomForest)
15. Evaluierung (ROC, PR, Confusion Matrix, Reports)
16. Feature Importance (RF)
17. Optional: Hyperparameter‑Tuning (GridSearchCV)
18. Modell‑Persistenz (joblib: Laden/Speichern)
19. Hinweise & nächste Schritte


## 1) Setup & Installation (optional)
Benötigte Pakete installieren – nur ausführen, wenn nötig.

In [ ]:
# Optional: Pakete installieren
# !pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn joblib


## 2) Imports & Konfiguration
Zentrale Bibliotheken laden und globale Einstellungen definieren.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, classification_report, roc_auc_score,
                             roc_curve, precision_recall_curve, average_precision_score)
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
import joblib

# Reproduzierbarkeit
RANDOM_STATE = 42

# Plot‑Ordner
os.makedirs('plots', exist_ok=True)

# Hilfsfunktion: Figuren sauber als PNG speichern
def save_fig(fig, name, dpi=150):
    path = os.path.join('plots', name)
    fig.savefig(path, bbox_inches='tight', dpi=dpi)
    plt.close(fig)


## 3) Daten laden
CSV einlesen (erwartet `creditcard.csv` im Arbeitsverzeichnis).

In [ ]:
DATA_PATH = "creditcard.csv"
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Datendatei nicht gefunden: {DATA_PATH}. Bitte Pfad überprüfen.")

df = pd.read_csv(DATA_PATH)
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")


## 4) Schnellüberblick
Erster Blick auf Struktur, Typen und Grundstatistiken.

In [ ]:
print("\n=== Kopf der Daten ===")
display(df.head())

print("\n=== Info ===")
df.info()

print("\n=== Deskriptive Statistik (numerisch) ===")
display(df.describe().T)

print("\nSpalten:", list(df.columns))


## 5) Datenqualität
Fehlende Werte, Duplikate, Datentypen.

In [ ]:
missing = df.isnull().sum()
print("\n=== Fehlende Werte pro Spalte (>0) ===")
display(missing[missing > 0])

dups = df.duplicated().sum()
print(f"\nAnzahl kompletter Duplikate: {dups}")

print("\nDatentypen:")
display(df.dtypes)


## 6) Zielverteilung & Klassenungleichgewicht
Im Fraud‑Datensatz typischerweise stark unausgeglichen.

In [ ]:
print("\n=== Zielverteilung (Counts) ===")
display(df['Class'].value_counts())

print("\n=== Zielverteilung (Anteile) ===")
display(df['Class'].value_counts(normalize=True))


## 7) Feature Engineering: `Hour`
Aus Sekunden‑Zeitstempel (`Time`) eine Stunden‑Bucket bilden.

In [ ]:
df['Hour'] = (df['Time'] // 3600).astype(int)
df.head()[['Time','Hour','Amount','Class']].head()


## 8) Visuelle EDA (Plots werden in `./plots` gespeichert)
- Zielverteilung (Balken)
- `Amount` (log‑Skala) gesamt & nach Klasse
- Transaktionen pro Stunde
- Boxplot `Amount` nach Klasse (log‑Skala)
- Korrelationsmatrix (V1..V28, Amount, Class)
- PCA‑2D‑Projektion
- Beispiel‑Verteilungen für V1..V6


In [ ]:
# 8.1 Zielverteilung
fig = plt.figure(figsize=(6,4))
sns.countplot(x='Class', data=df)
plt.title('Verteilung der Zielvariable (Class)')
save_fig(fig, 'target_distribution.png')

# 8.2 Amount‑Verteilungen (log1p)
fig = plt.figure(figsize=(10,5))
plt.hist(np.log1p(df['Amount']), bins=100)
plt.title('Log(Amount+1) – gesamte Verteilung')
plt.xlabel('log1p(Amount)')
save_fig(fig, 'amount_log_distribution.png')

fig = plt.figure(figsize=(10,5))
for label, subset in df.groupby('Class'):
    plt.hist(np.log1p(subset['Amount']), bins=100, histtype='step', label=f'Class {label}', alpha=0.8)
plt.legend()
plt.title('Log(Amount+1) nach Class')
plt.xlabel('log1p(Amount)')
save_fig(fig, 'amount_log_by_class.png')

# 8.3 Transaktionen pro Stunde
fig = plt.figure(figsize=(12,4))
plt.hist(df['Hour'], bins=range(int(df['Hour'].min()), int(df['Hour'].max())+2))
plt.title('Transaktionen pro Stunde (aggregiert)')
plt.xlabel('Hour')
save_fig(fig, 'transactions_per_hour.png')

# 8.4 Boxplot Amount nach Klasse (log‑Scale)
fig = plt.figure(figsize=(8,5))
# Matplotlib‑Boxplot (keine seaborn‑Abhängigkeit nötig)
data0 = df.loc[df['Class']==0, 'Amount']
data1 = df.loc[df['Class']==1, 'Amount']
plt.boxplot([np.log1p(data0), np.log1p(data1)], labels=['0','1'])
plt.title('Boxplot log1p(Amount) nach Class')
save_fig(fig, 'boxplot_amount_by_class.png')

# 8.5 Korrelationsmatrix
features = [c for c in df.columns if c not in ['Time','Hour','Class']]
corr = df[features + ['Class']].corr()
fig = plt.figure(figsize=(14,12))
sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Korrelationsmatrix (inkl. Class)')
save_fig(fig, 'correlation_matrix.png')

# 8.6 PCA‑2D Projektion (nur Visualisierung)
X_vis = df[features].values
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_vis)
fig = plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], c=df['Class'], alpha=0.6)
plt.title('PCA 2D Projektion')
plt.xlabel('PC1'); plt.ylabel('PC2')
save_fig(fig, 'pca_2d.png')

# 8.7 Beispiel‑Verteilungen V1..V6
for c in ['V1','V2','V3','V4','V5','V6']:
    fig = plt.figure(figsize=(8,4))
    for label, subset in df.groupby('Class'):
        plt.hist(subset[c], bins=80, histtype='step', alpha=0.8, label=f'Class {label}')
    plt.title(f'Distribution {c} nach Class')
    plt.legend()
    save_fig(fig, f'distr_{c}_by_class.png')


## 9) Kompakter Datenqualitäts‑Report

In [ ]:
quality_report = {
    'n_rows': int(df.shape[0]),
    'n_cols': int(df.shape[1]),
    'missing_counts': df.isnull().sum().to_dict(),
    'n_duplicates': int(df.duplicated().sum()),
    'class_counts': df['Class'].value_counts().to_dict(),
    'class_ratio': df['Class'].value_counts(normalize=True).to_dict()
}
quality_report


## 10) Modell‑Vorbereitung (Features & Ziel)
`Time` und `Hour` werden hier standardmäßig ausgeschlossen.

In [ ]:
X = df.drop(columns=['Class','Time','Hour'])
y = df['Class']
X.shape, y.shape


## 11) Train/Test Split (stratifiziert)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=RANDOM_STATE
)
X_train.shape, X_test.shape, y_train.value_counts().to_dict()


## 12) Skalierung (StandardScaler)
Skalieren getrennt für Train/Test (Leakage vermeiden).

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
X_train_scaled.shape, X_test_scaled.shape


## 13) Klassenbalancierung (SMOTE auf Trainingsdaten)
Gleicht Klassen aus, um lernbare Signale zu verstärken.

In [ ]:
print('Vor SMOTE (Train):')
print(y_train.value_counts())

sm = SMOTE(random_state=RANDOM_STATE)
X_res, y_res = sm.fit_resample(X_train_scaled, y_train)

print('\nNach SMOTE (Resampled Train):')
print(pd.Series(y_res).value_counts())

X_res.shape, pd.Series(y_res).value_counts(normalize=True).to_dict()


## 14) Baseline‑Modelle trainieren
- **Logistic Regression** als lineare Baseline
- **RandomForest** als nichtlinearer Ensembler

In [ ]:
models = {}

# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_res, y_res)
models['logreg'] = (lr, scaler)

# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_res, y_res)
models['rf'] = (rf, scaler)

list(models.keys())


## 15) Evaluierung
Berichte, ROC‑Kurven, Precision‑Recall, Confusion Matrix (werden gespeichert).

In [ ]:
def evaluate_model(model, scaler, X_test, y_test, name='model'):
    Xs = scaler.transform(X_test) if scaler is not None else X_test
    y_pred = model.predict(Xs)
    y_proba = model.predict_proba(Xs)[:,1] if hasattr(model, 'predict_proba') else model.decision_function(Xs)

    print(f"\n--- Auswertung: {name} ---")
    print(classification_report(y_test, y_pred, digits=4))
    auc = roc_auc_score(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    print(f"ROC AUC: {auc:.4f}")
    print(f"Average Precision (PR AUC): {ap:.4f}")

    # ROC
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    fig = plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr)
    plt.plot([0,1],[0,1], '--', linewidth=0.7)
    plt.xlabel('FPR'); plt.ylabel('TPR')
    plt.title(f'ROC - {name} (AUC={auc:.4f})')
    save_fig(fig, f'roc_{name}.png')

    # Precision‑Recall
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    fig = plt.figure(figsize=(6,5))
    plt.plot(recall, precision)
    plt.xlabel('Recall'); plt.ylabel('Precision')
    plt.title(f'PR - {name} (AP={ap:.4f})')
    save_fig(fig, f'pr_{name}.png')

    # Confusion Matrix
    from itertools import product
    cm = confusion_matrix(y_test, y_pred)
    fig = plt.figure(figsize=(5,4))
    plt.imshow(cm, aspect='auto')
    for (i,j) in product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, cm[i,j], ha='center', va='center')
    plt.xticks([0,1], ['Pred 0','Pred 1'])
    plt.yticks([0,1], ['True 0','True 1'])
    plt.title(f'Confusion Matrix - {name}')
    save_fig(fig, f'confmat_{name}.png')

for name, (model, sc) in models.items():
    evaluate_model(model, sc, X_test, y_test, name=name)


## 16) Feature Importance (RandomForest)
Ermittelt die wichtigsten Variablen gemäß RF‑Importances.

In [ ]:
rf_model = models['rf'][0]
importances = rf_model.feature_importances_
feat_names = X.columns
feat_imp = pd.DataFrame({'feature': feat_names, 'importance': importances}).sort_values('importance', ascending=False)
display(feat_imp.head(15))

fig = plt.figure(figsize=(8,6))
top = feat_imp.head(15)[::-1]
plt.barh(top['feature'], top['importance'])
plt.title('Top 15 Feature Importances (RF)')
save_fig(fig, 'rf_feature_importances_top15.png')


## 17) Optional: Hyperparameter‑Tuning (kleiner Grid)
Nur ausführen, wenn mehr Rechenzeit akzeptabel ist.

In [ ]:
# param_grid = {'n_estimators': [100, 200], 'max_depth': [None, 10, 20]}
# cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
# gs = GridSearchCV(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
#                   param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
# gs.fit(X_res, y_res)
# print('Beste Parameter:', gs.best_params_)
# joblib.dump(gs.best_estimator_, 'best_rf_gridsearch.joblib')


## 18) Modell‑Persistenz (Laden/Speichern mit joblib)
Speichert RF + Scaler + Featureliste – und lädt bei erneutem Lauf.

In [ ]:
MODEL_PATH = "rf_fraud_model.joblib"

if os.path.exists(MODEL_PATH):
    print("Lade bestehendes Modell...")
    saved = joblib.load(MODEL_PATH)
    rf_model = saved['model']
    scaler = saved['scaler']
    print("Modell geladen – kein erneutes Training erforderlich.")
else:
    print("Kein gespeichertes Modell gefunden. Speichere aktuelles RF‑Modell...")
    joblib.dump({'model': rf_model, 'scaler': scaler, 'features': list(X.columns)}, MODEL_PATH)
    print("Modell gespeichert:", MODEL_PATH)


## 19) Hinweise & nächste Schritte
- **Metriken:** Bei starkem Ungleichgewicht ist **Average Precision (PR‑AUC)** besonders aussagekräftig.

- **Thresholds:** Geschäftsziele definieren (Kosten FP vs. FN) und optimalen Schwellwert wählen.

- **Cross‑Validation:** In Produktion **imblearn‑Pipeline** verwenden und Resampling **innerhalb** der CV machen.

- **Kalibrierung:** Platt scaling / isotonic regression für kalibrierte Wahrscheinlichkeiten.

- **Monitoring:** Drift‑Detection, Retraining‑Trigger, A/B‑Tests einplanen.

